# NovaSeat Churn Prediction — Scoring Pipeline

Loads trained model artifacts, scores accounts from a CSV, and exports results
with churn probability, risk tier, and SHAP-based churn drivers.

**Prerequisites:** Run the training notebook first to generate model artifacts.

In [ ]:
!pip install -q pandas numpy scikit-learn xgboost shap joblib

In [ ]:
# ============================================================
# Sync repo from GitHub (auto clone or pull)
# ============================================================
import os

REPO_URL = "https://github.com/paoloneh/novaseat-v2.git"
REPO_DIR = "/content/novaseat-v2"

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

print(f"Repo synced at: {REPO_DIR}")
print(f"Colab files at: {REPO_DIR}/colab/")

In [ ]:
# ============================================================
# Configuration — edit these before running
# ============================================================
import os

# Auto-detect Colab Enterprise (headless Vertex AI execution)
IS_COLAB_ENTERPRISE = os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE"

USE_DRIVE = not IS_COLAB_ENTERPRISE  # Drive is not supported in Colab Enterprise
DRIVE_ARTIFACTS_PATH = "/content/drive/MyDrive/novaseat-model/artifacts"
LOCAL_ARTIFACTS_PATH = "/content/artifacts"  # local working dir for GCS downloads
GCS_ARTIFACTS_PATH = "gs://novaseat-churn-colab/artifacts"  # GCS fallback for Enterprise
GCS_INPUT_CSV = "gs://novaseat-churn-colab/scoring-input/accounts_export.csv"  # DB export from n8n

COMPUTE_SHAP = True       # Set False to skip SHAP driver computation (faster)
DRY_RUN = False           # Set True to only print results, not save

RISK_TIERS = {
    "Low": (0.0, 0.3),
    "Medium": (0.3, 0.5),
    "High": (0.5, 0.7),
    "Critical": (0.7, 1.0),
}

if IS_COLAB_ENTERPRISE:
    print("Running in Colab Enterprise — using GCS for artifacts")
else:
    print("Running in interactive Colab — using Google Drive for artifacts")

In [ ]:
# ============================================================
# Load model artifacts (Drive or GCS depending on environment)
# ============================================================
import os
import json
import subprocess
import zipfile
from pathlib import Path

import joblib

if IS_COLAB_ENTERPRISE:
    # Download artifacts from GCS to local dir
    os.makedirs(LOCAL_ARTIFACTS_PATH, exist_ok=True)
    subprocess.run(
        ["gcloud", "storage", "cp", "-r", f"{GCS_ARTIFACTS_PATH}/*", LOCAL_ARTIFACTS_PATH],
        check=True,
    )
    ARTIFACTS_DIR = LOCAL_ARTIFACTS_PATH
elif USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACTS_DIR = DRIVE_ARTIFACTS_PATH
else:
    # Upload the artifacts zip from the training notebook
    from google.colab import files
    uploaded = files.upload()  # select novaseat_artifacts.zip
    zip_name = list(uploaded.keys())[0]
    os.makedirs(LOCAL_ARTIFACTS_PATH, exist_ok=True)
    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(LOCAL_ARTIFACTS_PATH)
    ARTIFACTS_DIR = LOCAL_ARTIFACTS_PATH


def load_artifacts(artifacts_dir: str):
    """Load model, scaler, and feature columns from the artifacts directory."""
    d = Path(artifacts_dir)
    model = joblib.load(d / "model.joblib")
    scaler = joblib.load(d / "scaler.joblib")
    with open(d / "feature_columns.json") as f:
        feature_columns = json.load(f)
    print(f"Loaded model artifacts from {d}/")
    return model, scaler, feature_columns


model, scaler, feature_columns = load_artifacts(ARTIFACTS_DIR)

In [ ]:
# ============================================================
# Imports and constants
# ============================================================
import logging
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

TREND_MAP = {"Declining": -1, "Stable": 0, "Increasing": 1}
PLAN_TYPES = {"Starter", "Pro", "Enterprise"}
TIERS = {"Basic", "Premium", "Free"}

In [ ]:
# ============================================================
# Load accounts CSV to score
# ============================================================
import subprocess

if IS_COLAB_ENTERPRISE:
    # Download fresh account data exported by n8n from the database
    CSV_PATH = "/content/accounts_export.csv"
    subprocess.run(
        ["gcloud", "storage", "cp", GCS_INPUT_CSV, CSV_PATH],
        check=True,
    )
else:
    # Interactive mode: fall back to seed CSV for dev/testing
    CSV_PATH = DRIVE_ARTIFACTS_PATH + "/accounts_seed.csv"

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} accounts from {CSV_PATH}")
df.head()

In [ ]:
# ============================================================
# Prepare features
# ============================================================

def prepare_features(df: pd.DataFrame, feature_columns: list) -> pd.DataFrame:
    """Transform raw DB columns into the model's expected feature matrix."""
    feat = df.copy()

    # Encode trend if it's a string
    if feat["events_per_month_trend"].dtype == object:
        feat["events_per_month_trend"] = feat["events_per_month_trend"].map(TREND_MAP).fillna(0)

    # Encode boolean columns
    for col in ["has_dedicated_csm"]:
        if feat[col].dtype == object or feat[col].dtype == bool:
            feat[col] = feat[col].astype(int)

    # One-hot encode plan_type (Starter as baseline)
    if "plan_type" in feat.columns:
        feat["plan_Pro"] = (feat["plan_type"] == "Pro").astype(int)
        feat["plan_Enterprise"] = (feat["plan_type"] == "Enterprise").astype(int)

    # One-hot encode platform_tier (Basic as baseline)
    if "platform_tier" in feat.columns:
        feat["tier_Premium"] = (feat["platform_tier"] == "Premium").astype(int)
        feat["tier_Free"] = (feat["platform_tier"] == "Free").astype(int)
    else:
        feat["tier_Premium"] = 0
        feat["tier_Free"] = 0

    # Ensure payment_auto exists
    if "payment_auto" not in feat.columns:
        feat["payment_auto"] = 0

    # Ensure all extra features exist with defaults
    for col in ["senior_citizen", "has_partner", "has_dependents",
                "paperless_billing", "online_security", "online_backup", "streaming_tv"]:
        if col not in feat.columns:
            feat[col] = 0

    # Select and order features
    missing = [c for c in feature_columns if c not in feat.columns]
    if missing:
        logger.error("Missing feature columns in data: %s", missing)
        raise ValueError(f"Missing feature columns: {missing}")

    X = feat[feature_columns].fillna(0)
    return X

In [ ]:
# ============================================================
# Score accounts
# ============================================================

def assign_tier(prob: float) -> str:
    for tier, (lo, hi) in RISK_TIERS.items():
        if lo <= prob < hi:
            return tier
    return "Critical"


def score_accounts(
    df: pd.DataFrame,
    model,
    scaler,
    feature_columns: list,
    compute_drivers: bool = True,
) -> pd.DataFrame:
    """Score all accounts and return DataFrame with predictions."""
    X = prepare_features(df, feature_columns)
    X_scaled = pd.DataFrame(scaler.transform(X), columns=X.columns, index=X.index)

    # Predict probabilities
    probabilities = model.predict_proba(X_scaled)[:, 1]
    tiers = [assign_tier(p) for p in probabilities]

    # SHAP drivers
    drivers_json = [None] * len(df)
    if compute_drivers:
        try:
            import shap
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(X_scaled)
            for i in range(len(X_scaled)):
                row_shap = shap_values[i]
                top_idx = np.argsort(np.abs(row_shap))[-3:][::-1]
                drivers = [
                    {"driver": X_scaled.columns[idx], "impact": round(float(row_shap[idx]), 4)}
                    for idx in top_idx
                ]
                drivers_json[i] = json.dumps(drivers)
        except ImportError:
            logger.warning("shap not installed, skipping driver computation")

    # Build result
    result = df.copy()
    result["churn_probability"] = probabilities.round(4)
    result["risk_tier"] = tiers
    result["churn_drivers"] = drivers_json
    result["last_scored_at"] = datetime.now(timezone.utc).isoformat()

    logger.info("Scored %d accounts", len(result))
    return result


scored = score_accounts(df, model, scaler, feature_columns, compute_drivers=COMPUTE_SHAP)

# Show tier distribution
print("\n=== Risk Tier Distribution ===")
tier_counts = scored["risk_tier"].value_counts()
for tier in ["Low", "Medium", "High", "Critical"]:
    count = tier_counts.get(tier, 0)
    print(f"  {tier}: {count} ({count / len(scored) * 100:.1f}%)")

scored.head(10)

In [ ]:
# ============================================================
# Explore results
# ============================================================

# High-risk accounts
high_risk = scored[scored["risk_tier"].isin(["High", "Critical"])]
print(f"High/Critical risk accounts: {len(high_risk)} of {len(scored)}")

display_cols = [c for c in ["name", "churn_probability", "risk_tier", "churn_drivers"] if c in scored.columns]
high_risk[display_cols].head(20)

In [ ]:
# ============================================================
# Export results
# ============================================================

# Only export the columns needed for the DB update
output_cols = ["name", "churn_probability", "risk_tier", "churn_drivers", "last_scored_at"]
export_df = scored[output_cols]

if DRY_RUN:
    print("Dry run — no file written.")
    print(f"\nTotal: {len(export_df)} accounts scored")
elif IS_COLAB_ENTERPRISE:
    # Save locally then upload to GCS
    local_out = "/content/scored_accounts.csv"
    export_df.to_csv(local_out, index=False)
    gcs_out = GCS_ARTIFACTS_PATH + "/scored_accounts.csv"
    subprocess.run(["gcloud", "storage", "cp", local_out, gcs_out], check=True)
    print(f"Uploaded scored accounts to GCS: {gcs_out}")
elif USE_DRIVE:
    out_path = DRIVE_ARTIFACTS_PATH + "/scored_accounts.csv"
    export_df.to_csv(out_path, index=False)
    print(f"Saved scored accounts to Google Drive: {out_path}")
else:
    out_path = "/content/scored_accounts.csv"
    export_df.to_csv(out_path, index=False)
    from google.colab import files as colab_files
    colab_files.download(out_path)
    print(f"Downloaded: {out_path}")